# Scenario: Ramp up offshore wind capacity

### 1.1. Import Libraries

In [1]:
import pypsa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import cartopy.crs as ccrs

### 1.2. Load Network Data
Load the solved network model from its `.nc` file.

In [2]:
config_name = "germany_base"
cluster = "450"

home = "/home/lucakristin/Desktop/my_pypsa"

# Path to the network file before solving
network_path_unsolved = f"{home}/pypsa-eur/resources/{config_name}/networks/base_s_{cluster}_elec.nc"
n_unsolved = pypsa.Network(network_path_unsolved)


# Path to the network file after solving
network_path_solved = f"{home}/pypsa-eur/results/{config_name}/networks/base_s_{cluster}_elec_.nc"
n = pypsa.Network(network_path_solved)

# Copy for scenario
n_scenario = n.copy()

INFO:pypsa.network.io:New version 1.2.0 available! (Current: 1.1.2)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, lines, links, loads, stores, sub_networks
INFO:pypsa.network.io:New version 1.2.0 available! (Current: 1.1.2)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, lines, links, loads, stores, sub_networks


### 1.3 Identify offshore carrier and generators in the south

In [3]:
n_scenario.generators.carrier.value_counts()

carrier
solar         450
onwind        450
OCGT          402
biomass       401
oil           374
CCGT          248
waste         134
coal           32
offwind-ac     31
lignite        25
offwind-dc      5
geothermal      4
Name: count, dtype: int64

#### Offshore

In [4]:
offshore = n_scenario.generators.index[n_scenario.generators.carrier.isin(["offwind-ac", "offwind-dc", "Wind Offshore"])]
n_scenario.generators.loc[offshore, "p_nom"] *= 1.2

In [5]:
print("Summed capacity for scenario:")
print(n_scenario.generators.loc[offshore, "p_nom"].sum())

Summed capacity for scenario:
13396.491600000001


In [6]:
print("Summed capacity for base:")
print(n.generators.loc[offshore, "p_nom"].sum())

Summed capacity for base:
11163.743


#### Fossil fuels

In [7]:
fossil_carriers = ['CCGT', 'gas', 'lignite', 'hard coal', 'coal', 'oil']
fossil_gens = n_scenario.generators[n_scenario.generators.carrier.isin(fossil_carriers)]
print(fossil_gens[['bus', 'carrier', 'p_nom']])

                  bus carrier       p_nom
name                                     
DE0 0 oil       DE0 0     oil     4.16800
DE0 1 oil       DE0 1     oil     0.78320
DE0 10 CCGT    DE0 10    CCGT     0.25000
DE0 10 oil     DE0 10     oil     8.83270
DE0 101 CCGT  DE0 101    CCGT    11.62266
...               ...     ...         ...
DE0 98 CCGT    DE0 98    CCGT  1910.68700
DE0 98 oil     DE0 98     oil   148.15180
DE0 99 CCGT    DE0 99    CCGT     9.31700
DE0 99 coal    DE0 99    coal  1028.00000
DE0 99 oil     DE0 99     oil    50.36937

[679 rows x 3 columns]


In [8]:
south_buses = n_scenario.buses.index[n_scenario.buses.v_nom >= 380 & (n_scenario.buses.y >= 47.5)]
south_buses

Index(['DE0 0', 'DE0 1', 'DE0 10', 'DE0 100', 'DE0 101', 'DE0 102', 'DE0 103',
       'DE0 104', 'DE0 105', 'DE0 106',
       ...
       'DE0 90 battery', 'DE0 91 battery', 'DE0 92 battery', 'DE0 93 battery',
       'DE0 94 battery', 'DE0 95 battery', 'DE0 96 battery', 'DE0 97 battery',
       'DE0 98 battery', 'DE0 99 battery'],
      dtype='object', name='name', length=1350)

In [9]:
# Select southern fossil generators
south_fossil_gens = n_scenario.generators[
    (n_scenario.generators.carrier.isin(fossil_carriers)) &
    (n_scenario.generators.bus.isin(south_buses))
].index

print(f"Southern fossil generators: {len(south_fossil_gens)}")
print(n_scenario.generators.loc[south_fossil_gens, ['bus', 'carrier', 'p_nom']])

Southern fossil generators: 679
                  bus carrier       p_nom
name                                     
DE0 0 oil       DE0 0     oil     4.16800
DE0 1 oil       DE0 1     oil     0.78320
DE0 10 CCGT    DE0 10    CCGT     0.25000
DE0 10 oil     DE0 10     oil     8.83270
DE0 101 CCGT  DE0 101    CCGT    11.62266
...               ...     ...         ...
DE0 98 CCGT    DE0 98    CCGT  1910.68700
DE0 98 oil     DE0 98     oil   148.15180
DE0 99 CCGT    DE0 99    CCGT     9.31700
DE0 99 coal    DE0 99    coal  1028.00000
DE0 99 oil     DE0 99     oil    50.36937

[679 rows x 3 columns]


In [10]:
# reduce fossil fuels production in the south by 20%
n_scenario.generators.loc[south_fossil_gens, 'p_nom'] *= 0.8

In [11]:
print("Summed capacity for southern fossil generators after reduction:")
print(n_scenario.generators.loc[south_fossil_gens, 'p_nom'].sum())

Summed capacity for southern fossil generators after reduction:
61021.1634528


In [12]:
print("Summed capacity for southern fossil generators in base scenario:")
print(n.generators.loc[south_fossil_gens, 'p_nom'].sum())

Summed capacity for southern fossil generators in base scenario:
76276.45431599999


In [17]:
config_name_scenario = "germany_scenario"

In [ ]:
# Save the modified network to the path that snakemake will use as input
# This will overwrite the existing base network file in the resources directory
# You might want to back up the original file first

unsolved_path_for_snakemake = f"{home}/pypsa-eur/resources/{config_name_scenario}/networks/base_s_{cluster}_elec_.nc"
n_scenario.export_to_netcdf(unsolved_path_for_snakemake)
print(f"Scenario network saved to {unsolved_path_for_snakemake}")


INFO:pypsa.network.io:Exported network 'Unnamed Network' saved to '/home/lucakristin/Desktop/my_pypsa/pypsa-eur/resources/germany_scenario/networks/base_s_450_elec_.nc contains: sub_networks, buses, loads, lines, links, generators, carriers, stores


Scenario network saved to /home/lucakristin/Desktop/my_pypsa/pypsa-eur/resources/germany_scenario/networks/base_s_450_elec_.nc


In [ ]:
# Solve scenario
#n_scenario.optimize()

# Save scenario results
#n_scenario.export_to_netcdf("scenario_1.nc")

/tmp/ipykernel_6185/315664858.py:2: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n_scenario.optimize()
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 10/10 [00:00<00:00, 256.55it/s]
INFO:linopy.io: Writing time: 0.36s


Running HiGHS 1.14.0 (git hash: n/a): Copyright (c) 2026 under MIT licence terms
LP linopy-problem-pl7r8vkt has 347206 rows; 167731 cols; 732419 nonzeros
Coefficient ranges:
  Matrix  [1e-02, 6e+01]
  Cost    [9e-03, 1e+03]
  Bound   [6e+07, 6e+07]
  RHS     [8e-02, 3e+04]
Presolving model
161104 rows, 158342 cols, 516117 nonzeros 0s
136968 rows, 134206 cols, 475939 nonzeros 0s
Dependent equations search running on 34162 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.01s (limit = 1000.00s)
136968 rows, 134206 cols, 475939 nonzeros 1s
Presolve reductions: rows 136968(-210238); columns 134206(-33525); nonzeros 475939(-256480) 
Solving the presolved LP
Using dual simplex solver
  Iteration        Objective     Infeasibilities num(sum)
          0    -6.0002860720e+01 Ph1: 62349(1.47453e+08); Du: 8022(60.0029) 1.3s
      22060     3.0654478643e+05 Pr: 12287(1.69313e+09); Du: 0(3.30446e-08) 7.7s
      23178     3.1561379969e+05 Pr: 12472(

KeyboardInterrupt: 

In [18]:
network_path_scenario_solved = f"{home}/pypsa-eur/results/{config_name_scenario}/networks/base_s_{cluster}_elec_.nc"
n_scenario = pypsa.Network(network_path_scenario_solved)

FileNotFoundError: [Errno 2] No such file or directory: '/home/lucakristin/Desktop/my_pypsa/pypsa-eur/results/germany_scenario/networks/base_s_450_elec_.nc'

### 2.2. Static Network Plot with Load Distribution

#### Base Scenario

In [ ]:
snapshot = n.snapshots[0]
load_distribution = n.loads_t.p_set.loc[snapshot].groupby(n.loads.bus).sum()


# Calculate total flow for line/link widths
line_flow = n.lines_t.p0.sum(axis=0)
link_flow = n.links_t.p0.sum(axis=0)

bus_size_factor = 1e9
branch_width_factor = 10000

# Select a snapshot to analyze
now = n.snapshots[9]
loading = n.lines_t.p0.loc[now] / n.lines.s_nom
print(f"Line loading statistics for snapshot {now}:")
display(loading.describe())


# Create a new figure for this plot
fig, ax = plt.subplots(
    1, 1, figsize=(8, 8),
    subplot_kw={"projection": ccrs.EqualEarth()}
)

n.plot(
    ax=ax,
    geomap=True,
    line_colors=loading.abs(),
    bus_sizes=(load_distribution/20000),
    bus_alpha=0.8,
    bus_split_circle=True,
    line_width=line_flow / branch_width_factor,
    link_width=link_flow / branch_width_factor,
    title="Energy Balance by Bus"
)


### Ramped up Offshore Scenario

In [ ]:
snapshot = n_scenario.snapshots[0]
load_distribution = n_scenario.loads_t.p_set.loc[snapshot].groupby(n_scenario.loads.bus).sum()


# Calculate total flow for line/link widths
line_flow = n_scenario.lines_t.p0.sum(axis=0)
link_flow = n_scenario.links_t.p0.sum(axis=0)

bus_size_factor = 1e9
branch_width_factor = 10000

# Select a snapshot to analyze
now = n_scenario.snapshots[9]
loading = n_scenario.lines_t.p0.loc[now] / n_scenario.lines.s_nom
print(f"Line loading statistics for snapshot {now}:")
display(loading.describe())


# Create a new figure for this plot
fig, ax = plt.subplots(
    1, 1, figsize=(8, 8),
    subplot_kw={"projection": ccrs.EqualEarth()}
)

n_scenario.plot(
    ax=ax,
    geomap=True,
    line_colors=loading.abs(),
    bus_sizes=(load_distribution/20000),
    bus_alpha=0.8,
    bus_split_circle=True,
    line_width=line_flow / branch_width_factor,
    link_width=link_flow / branch_width_factor,
    title="Energy Balance by Bus"
)


### 3.1. Top Buses by Annual Load
A summary of the buses with the highest total electricity consumption over the simulation period.

In [ ]:
if not n.loads.empty and hasattr(n.loads_t, "p_set") and not n.loads_t.p_set.empty:
    annual_load_by_bus = n.loads_t.p_set.sum().sort_values(ascending=False)
    print("\nTop buses by annual load [MWh over snapshots]:")
    display(annual_load_by_bus.head(15).to_frame("load_MWh"))

### 3.2. Total Load Over Simulation Period
The total aggregated load for the entire network over all snapshots.

In [ ]:
# Total load over the simulation period
loads = n.loads_t.p_set.sum().to_frame("total_load_MWh") #sum over all snapshots
print("Total load summary:")
loads

---
## 4. Generation Analysis
This section visualizes the generation side of the network, including installed capacities, dispatch patterns, and curtailment.

In [ ]:
#calculated voltage phase angle for every bus in your network at every time snapshot
phase_angles = n.buses_t.v_ang
phase_angles.head()

### 4.1. Installed Capacity Analysis
An overview of the installed generation capacity, grouped by carrier type and by bus.

In [ ]:
if not n.generators.empty:
    cap_by_carrier = n.generators.groupby("carrier")["p_nom"].sum().sort_values(ascending=False)
    print("Installed generator capacity by carrier [MW]:")
    display(cap_by_carrier.head(20).to_frame("p_nom_MW"))

    cap_by_bus = n.generators.groupby("bus")["p_nom"].sum().sort_values(ascending=False)
    print("\nTop buses by installed generation capacity [MW]:")
    display(cap_by_bus.head(15).to_frame("p_nom_MW"))

### 4.2. Average Generator Availability per Carrier
The following plots show the average `p_max_pu` (per unit maximum availability) for all generators of a specific carrier type. This is particularly relevant for renewable sources like wind and solar.

In [ ]:
# Get all unique carriers
carriers = n.generators.carrier.unique()

# Create a figure with subplots for each carrier
n_carriers = len(carriers)
n_cols = 3
n_rows = (n_carriers + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4), sharex=True)
axes = axes.flatten() # Flatten the 2D array of axes for easy iteration

for i, carrier in enumerate(carriers):
    ax = axes[i]
    
    # Find all generators for the current carrier
    gens_for_carrier = n.generators.index[n.generators.carrier == carrier]
    
    # Find the intersection of generators for the carrier and columns in p_max_pu
    common_gens = n.generators_t.p_max_pu.columns.intersection(gens_for_carrier)
    
    # Plot the AVERAGE p_max_pu for these generators
    if not common_gens.empty:
        p_max_pu_carrier = n.generators_t.p_max_pu[common_gens]
        p_max_pu_carrier.mean(axis=1).plot(ax=ax, legend=False)
    
    ax.set_title(f"Average Availability for {carrier}")
    ax.set_ylabel("p_max_pu")
    ax.set_ylim(0, 1)

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

### 4.3. Power Dispatch by Carrier
This stacked area plot shows the total power generated (dispatched) by each major carrier type over time.

In [ ]:
p_by_carrier = n.generators_t.p.T.groupby(n.generators.carrier).sum().T
# Drop carriers with very low production for clarity
to_drop = p_by_carrier.max()[p_by_carrier.max() < 1700].index
p_by_carrier.drop(to_drop, axis=1, inplace=True)

# Define colors and order for the dispatch plot
colors = {
    "biomass": "green", "coal": "black", "lignite": "brown",
    "offwind-ac": "cyan", "onwind": "blue", "solar": "yellow",
    "gas": "orange", "oil": "gray", "ror": "turquoise", "geothermal": "red",
}
# Reorder columns for a conventional stack
cols = [
    "lignite", "coal", "biomass", "onwind", "offwind-ac", "solar",
]
p_by_carrier = p_by_carrier.reindex(columns=cols).fillna(0)

c = [colors.get(col, "gray") for col in p_by_carrier.columns]
fig, ax = plt.subplots(figsize=(12, 6))
p_by_carrier.div(1e3).plot(kind="area", ax=ax, lw=0, color=c, alpha=0.7)
ax.legend(ncol=3, loc="upper left", bbox_to_anchor=(0, 1.02, 1, 0.2), frameon=False)
ax.set_ylabel("Dispatch [GW]")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

### 4.4. Curtailment Analysis
Curtailment is the reduction in the output of a generator from what it could otherwise produce. Here we calculate and plot it for the 'onwind' carrier as an example.

In [ ]:
carrier = "onwind"

capacity = n.generators.groupby("carrier").sum().at[carrier, "p_nom"]
p_available = n.generators_t.p_max_pu.multiply(n.generators["p_nom"])
p_available_by_carrier = p_available.T.groupby(n.generators.carrier).sum().T
p_curtailed_by_carrier = p_available_by_carrier - p_by_carrier

p_df = pd.DataFrame(
    {
        carrier + " available": p_available_by_carrier[carrier],
        carrier + " dispatched": p_by_carrier[carrier],
        carrier + " curtailed": p_curtailed_by_carrier[carrier],
    }
)

p_df[carrier + " capacity"] = capacity

# Correct for any minor negative values from floating point inaccuracies
p_df.loc[p_df[f"{carrier} curtailed"] < 0, f"{carrier} curtailed"] = 0

fig, ax = plt.subplots(figsize=(12, 6))
p_df[[carrier + " dispatched", carrier + " curtailed"]].plot(kind="area", ax=ax, lw=0, alpha=0.7)
p_df[[carrier + " available", carrier + " capacity"]].plot(ax=ax, lw=2)

ax.set_xlabel("")
ax.set_ylabel("Power [MW]")
ax.set_title(f"Dispatch and Curtailment for {carrier.title()}")
ax.legend()
plt.tight_layout()
plt.show()